# 第 8 课：将强化学习接入二维圆柱绕流

**学习目标**：定义与算法无关的 flow-control 环境接口、奖励及评价流程。

假设圆柱绕流要减小时间平均阻力，同时限制作动幅值和变化率：

$$r_t=-\underbrace{(C_{D,t}-C_{D,\rm ref})}_{\text{减阻}}-\lambda_a a_t^2-\lambda_s(a_t-a_{t-1})^2.$$

这不是唯一方案。研究论文需说明：是否去掉 baseline、是否要惩罚升力波动、时间平均窗口有多长、物理单位与无量纲化、作动功耗是否应计入净节能。

**关键区别**：训练代理模型所用的流场/阻力监督标签，和用于挑选控制策略的“真实物理评价”，不能完全重合，否则极易发生代理奖励乐观偏差。

## 8.1 对齐不同时间尺度

一个 RL `step` 不一定对应一个 CFD 时间步。假设 CFD 使用 $\Delta t_{CFD}$，动作每 $N_c$ 个 CFD 步更新，那么

$$\Delta t_{control}=N_c\Delta t_{CFD},\qquad r_t\approx-\frac1{N_c}\sum_{j=1}^{N_c} C_D(t,j) - \text{penalties}.$$

实际要考虑：启动瞬态、流场传感器的采样频率、动作平滑、守恒条件与稳定边界条件。

In [ ]:
from dataclasses import dataclass
import numpy as np
import matplotlib.pyplot as plt

@dataclass
class RewardConfig:
    cd_reference: float = 1.40
    action_weight: float = 0.04
    smooth_weight: float = 0.03

def drag_reward(cd_samples, action, previous_action, config=RewardConfig()):
    cd_mean = np.asarray(cd_samples).mean()
    reduction = config.cd_reference - cd_mean
    control_cost = config.action_weight*action**2
    smooth_cost = config.smooth_weight*(action-previous_action)**2
    return float(reduction-control_cost-smooth_cost)

# 仅为奖励函数单元测试，合成阻力数据不是 CFD 结果
base_cd=np.full(20,1.40)
controlled_cd=np.full(20,1.25)
print('Uncontrolled reward:', drag_reward(base_cd,0.,0.))
print('Controlled reward:', drag_reward(controlled_cd,.5,.5))
assert drag_reward(controlled_cd,.5,.5) > drag_reward(base_cd,0.,0.)

## 8.2 Gymnasium 风格的 CFD 环境契约

真正实现 `CFDFlowEnv.step` 时应完成：应用动作边界条件 → 求解 `N_control` 步 → 提取压力/升阻力/流场观测 → 计算奖励 → 判断是否物理终止（求解发散等）或预算截断。

这里**只定义数据结构与契约**，没有偷偷用合成数据冒充圆柱 CFD。

In [ ]:
@dataclass
class Transition:
    observation: np.ndarray
    action: np.ndarray
    reward: float
    next_observation: np.ndarray
    terminated: bool
    truncated: bool

class CFDFlowEnvProtocol:
    """抽象的调用约定；子类必须接入物理求解器或代理模型。"""
    def reset(self, seed=None):
        raise NotImplementedError('Connect to CFD or learned surrogate')
    def step(self, action):
        raise NotImplementedError('Advance physical dynamics before returning reward')

t = Transition(np.zeros(8),np.array([0.1]),0.03,np.ones(8),False,False)
assert t.observation.shape == (8,)
print('One transition schema verified:', t)

## 8.3 真实流场控制的分层验证

| 验证级别 | 你应该记录的结果 |
|---|---|
| 物理求解器 | 质量守恒、边界条件、网格/时间步收敛、基线 $C_D$ / $C_L$ 与文献比较 |
| 代理模型 | teacher forcing 的单步误差 + free rollout 的长时误差 |
| RL 训练 | reward、价值误差、action 分布、KL、PPO clipping fraction |
| 独立物理检验 | 基线/控制平均阻力、升力波动、作动代价、多随机种子 |
| 泛化 | 未参与训练的 $Re$、初始条件、传感器扰动与模型误差 |

## 8.4 作业：毕设最小闭环

1. 不使用代理：让你自己的 JAX CFD 求解器接收常数控制指令，跑到可复现的阻力曲线。
2. 根据圆柱周围有限个压力探针，构造观测向量并明确 normalization。
3. 用少量 CFD 数据训练动力学 surrogate，先做开环多步预测。
4. PPO 只在 surrogate 上训练，每隔固定训练预算提取策略回到 CFD 测试。
5. 汇报 **CFD 环境中真实阻力**，而不仅是代理环境中的预测奖励。

延伸：[HydroGym 官方项目](https://dynamicslab.github.io/hydrogym/) 与 [JAX 后端教程](https://dynamicslab.github.io/hydrogym/docs/examples/jax/getting_started/)。